In [5]:
import pandas as pd
import numpy as np

# visualization (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# preprocessing
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# models
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression

# polynomial
from sklearn.preprocessing import PolynomialFeatures

# evaluation
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score


import warnings
warnings.filterwarnings("ignore")

In [6]:
df = pd.read_csv(r"C:\Users\Parth\Documents\mldatset.csv")

In [7]:
df


,Loan_ID,Gender,Married,Dependents,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,No,5849,0.0,$146,360,1,Urban,Y
1,LP001003,Male,Yes,1,No,4583,1508.0,$128,360,1,Rural,N
2,LP001005,Male,Yes,0,Yes,3000,0.0,$66,360,1,Urban,Y
3,LP001006,Male,Yes,0,No,2583,2358.0,$120,360,1,Urban,Y
4,LP001008,Male,No,0,No,6000,0.0,$141,360,1,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,No,2900,0.0,$71,360,1,Rural,Y
610,LP002979,Male,Yes,3,No,4106,0.0,$40,180,1,Rural,Y
611,LP002983,Male,Yes,1,No,8072,240.0,$253,360,1,Urban,Y
612,LP002984,Male,Yes,2,No,7583,0.0,$187,360,1,Urban,Y


In [8]:
df = df.drop([ 'CoapplicantIncome', 'Loan_Amount_Term'], axis=1)


In [9]:
df.isnull().sum()

Loan_ID            0
Gender             0
Married            0
Dependents         0
Self_Employed      0
ApplicantIncome    0
LoanAmount         0
Credit_History     0
Property_Area      0
Loan_Status        0
dtype: int64

In [10]:
df

,Loan_ID,Gender,Married,Dependents,Self_Employed,ApplicantIncome,LoanAmount,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,No,5849,$146,1,Urban,Y
1,LP001003,Male,Yes,1,No,4583,$128,1,Rural,N
2,LP001005,Male,Yes,0,Yes,3000,$66,1,Urban,Y
3,LP001006,Male,Yes,0,No,2583,$120,1,Urban,Y
4,LP001008,Male,No,0,No,6000,$141,1,Urban,Y
...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,No,2900,$71,1,Rural,Y
610,LP002979,Male,Yes,3,No,4106,$40,1,Rural,Y
611,LP002983,Male,Yes,1,No,8072,$253,1,Urban,Y
612,LP002984,Male,Yes,2,No,7583,$187,1,Urban,Y


In [11]:
# Drop Loan_ID (important)
if "Loan_ID" in df.columns:
    df.drop("Loan_ID", axis=1, inplace=True)

# Fill missing values
df.fillna(df.mode().iloc[0], inplace=True)


# ===============================
# ENCODING
# ===============================
le = LabelEncoder()

for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])


# ===============================
# 🔹 CLASSIFICATION MODEL
# ===============================
X_cls = df.drop("Loan_Status", axis=1)
y_cls = df["Loan_Status"]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_cls, y_train_cls)

y_pred_cls = clf.predict(X_test_cls)
print("Classification Accuracy:", accuracy_score(y_test_cls, y_pred_cls))


# ===============================
# 🔹 REGRESSION MODEL
# ===============================
X_reg = df.drop("LoanAmount", axis=1)
y_reg = df["LoanAmount"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

reg = LinearRegression()
reg.fit(X_train_reg, y_train_reg)

y_pred_reg = reg.predict(X_test_reg)

print("MSE:", mean_squared_error(y_test_reg, y_pred_reg))
print("R2 Score:", r2_score(y_test_reg, y_pred_reg))

import pickle

pickle.dump(clf, open("clf.pkl", "wb"))
pickle.dump(reg, open("reg.pkl", "wb"))
pickle.dump(X_cls.columns, open("cols_cls.pkl", "wb"))
pickle.dump(X_reg.columns, open("cols_reg.pkl", "wb"))
# ===============================
# 🔥 USER INPUT
# ===============================
print("\n===== ENTER USER DETAILS =====")

gender = input("Gender (Male/Female): ")
married = input("Married (Yes/No): ")
dependents = input("Dependents (0/1/2/3+): ")

self_emp = input("Self Employed (Yes/No): ")
income = float(input("Applicant Income: "))

credit = int(input("Credit History (0/1): "))
property_area = input("Property Area (Urban/Rural/Semiurban): ")


# ===============================
# 🔄 MANUAL ENCODING
# ===============================
gender = 1 if gender == "Male" else 0
married = 1 if married == "Yes" else 0

self_emp = 1 if self_emp == "Yes" else 0

if dependents == "3+":
    dependents = 3
else:
    dependents = int(dependents)

if property_area == "Urban":
    property_area = 2
elif property_area == "Semiurban":
    property_area = 1
else:
    property_area = 0


# ===============================
# 📊 CLASSIFICATION INPUT
# ===============================
user_data_cls = pd.DataFrame([{
    'Gender': gender,
    'Married': married,
    'Dependents': dependents,
   
    'Self_Employed': self_emp,
    'ApplicantIncome': income,
    
    'LoanAmount': 100,  # dummy
  
    'Credit_History': credit,
    'Property_Area': property_area
}])

# Match columns
user_data_cls = user_data_cls.reindex(columns=X_cls.columns, fill_value=0)


# ===============================
# 📊 REGRESSION INPUT
# ===============================
user_data_reg = pd.DataFrame([{
    'Gender': gender,
    'Married': married,
    'Dependents': dependents,
   
    'Self_Employed': self_emp,
    'ApplicantIncome': income,
  
    'Credit_History': credit,
    'Property_Area': property_area,
    'Loan_Status': 1  # dummy
}])

# Match columns
user_data_reg = user_data_reg.reindex(columns=X_reg.columns, fill_value=0)


# ===============================
# 🔮 PREDICTION
# ===============================
loan_status = clf.predict(user_data_cls)
loan_amount = reg.predict(user_data_reg)


# ===============================
# 🎯 OUTPUT
# ===============================
print("\n===== RESULT =====")

if loan_status[0] == 1:
    print("Loan Approved ✅")
else:
    print("Loan Not Approved ❌")

print("Predicted Loan Amount:", int(loan_amount[0]))

Classification Accuracy: 0.7886178861788617
MSE: 4376.998225696894
R2 Score: 0.03730059113514139

===== ENTER USER DETAILS =====

===== RESULT =====
Loan Approved ✅
Predicted Loan Amount: 74


In [12]:
df


,Gender,Married,Dependents,Self_Employed,ApplicantIncome,LoanAmount,Credit_History,Property_Area,Loan_Status
0,1,0,0,0,5849,46,1,2,1
1,1,1,1,0,4583,28,1,0,0
2,1,1,0,1,3000,174,1,2,1
3,1,1,0,0,2583,20,1,2,1
4,1,0,0,0,6000,41,1,2,1
...,...,...,...,...,...,...,...,...,...
609,0,0,0,0,2900,178,1,0,1
610,1,1,3,0,4106,144,1,0,1
611,1,1,1,0,8072,111,1,2,1
612,1,1,2,0,7583,80,1,2,1
